# Step 1: Scaled Dot-Product Attention


$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$

## 数式とコードの対応

実装は Multi-Head Attention から呼ぶ前提で 4D テンソル `(batch, heads, seq_len, d_k)` を扱う。
論文の式は次元に依存しない(`matmul` は先頭次元をブロードキャストする)ので、先頭に `heads` が増えても同じ式がそのまま成り立つ。

| 数式 | コード | 形状 |
|---|---|---|
| $Q$ | `Q` | `(batch, heads, seq_q, d_k)` |
| $K$ | `K` | `(batch, heads, seq_k, d_k)` |
| $V$ | `V` | `(batch, heads, seq_k, d_v)` |
| $K^\top$ | `K.transpose(-2, -1)` | `(batch, heads, d_k, seq_k)` |
| $Q K^\top$ | `torch.matmul(Q, K.transpose(-2, -1))` | `(batch, heads, seq_q, seq_k)` |
| $\sqrt{d_k}$ | `math.sqrt(d_k)` | scalar |
| (mask 適用) | `scores.masked_fill(mask == 0, float('-inf'))` | `(batch, heads, seq_q, seq_k)` |
| $\mathrm{softmax}(\cdot)$ | `F.softmax(scores, dim=-1)` | `(batch, heads, seq_q, seq_k)` |
| $\mathrm{softmax}(\cdot)\, V$ | `torch.matmul(attn_weights, V)` | `(batch, heads, seq_q, d_v)` |

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):

    
    d_k = Q.size(-1) 

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k) # スコア行列: (batch, heads, seq_len, seq_len)
    
    # マスク適用（未来のトークンやパディングを無視）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)   # softmaxで重みを正規化　重要度　注意付き重み
    
    output = torch.matmul(attn_weights, V)  # 重み付き和
    return output, attn_weights



    




# テスト: バッチ2、ヘッド1、系列長4、次元8
batch, heads, seq_len, d_k = 2, 1, 4, 8
Q = torch.randn(batch, heads, seq_len, d_k)
K = torch.randn(batch, heads, seq_len, d_k)
V = torch.randn(batch, heads, seq_len, d_k)

output, attn_weights = scaled_dot_product_attention(Q, K, V)
print(output.shape)
print(attn_weights.shape)



#テスト：マスク付き
mask = torch.tensor([[[[1, 1, 1, 0]]],
                    [[[1, 1, 0, 0]]]])

output_m, attn_weights_m = scaled_dot_product_attention(Q, K, V, mask)
print(attn_weights_m[0, 0])   # 4列目が0
print(attn_weights_m[1, 0])   # 3,4列目が0

## 補足

- $\sqrt{d_k}$ で割る理由: 内積が大きいと softmax が飽和して勾配が消えるため


## 詰まったところ
K.transpose(-2, -1)が何を意味しているのかなかなか理解できなかった

## 学んだこと
transpose(-2, -1)は最後の2次元を入れ替えてるtranspose(-2, -1)
